# SQ-GAN DIV2K + AWGN/Rayleigh adaptation

This notebook retains the author model and training stages from `frapez1/SQ-GAN` at a pinned commit. The DIV2K pseudo-segmentation inputs and analog noisy channel are a new research adaptation, not the authors’ Cityscapes source-codec experiment. The original model communicates both a semantic stream and an image stream. The adapter applies the selected channel to each stream’s straight-through VQ output. It records latent complex channel uses separately from mask/index side information, which is not included in analog CBR.

The compatibility gate checks the legacy Lightning APIs, confirms mounted DIV2K/Kodak images, builds the model, and runs one synthetic forward/backward smoke pass before the long training stages. `RUN_MODE="full"` is selected below. Full mode runs the author’s three 150-epoch stages for each channel (900 stage-epochs total).


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, time, hashlib, shutil, platform

RUN_MODE = "full"  # full training: 150 epochs x 3 stages x 2 channels
EPOCHS_PER_STAGE = 150
SESSION_HOURS = 9.5
SESSION_DEADLINE_UNIX = time.time() + SESSION_HOURS * 3600
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"  # legacy author torch.load behavior
STOP_AFTER_STAGE_EPOCHS = 98  # preserve the total stage target; session deadline handles chunking
CHANNELS = ["awgn", "rayleigh"]
SNR_DB = 10.0
PERFECT_CSI = True  # explicit Rayleigh adaptation assumption
IMAGE_HEIGHT = 256
IMAGE_WIDTH = 512
BATCH_SIZE = 1  # preserve author config

REPO = Path("/tmp/SQ-GAN")  # source, top-k checkpoints and images stay outside exported output
OUTPUT = Path("/kaggle/working/sqgan_div2k_channel_results")
DIV2K_TRAIN = Path("/kaggle/input/notebooks/jagan028/div2k-dataset-generation-for-isr/Training/HR/hr_images")
DIV2K_VAL = Path("/kaggle/input/notebooks/jagan028/div2k-dataset-generation-for-isr/Validation/HR/hr_images")
KODAK_ROOT = Path("/kaggle/input/datasets/sherylmehta/kodak-dataset")
UPSTREAM_URL = "https://github.com/frapez1/SQ-GAN.git"
UPSTREAM_COMMIT = "0e5cdd7c0240b89749dad684645c53f783ac24cc"
BASE_CONFIG = "config/sqgan_cityscapes.yml"
OUTPUT.mkdir(parents=True, exist_ok=True)
print(json.dumps({"python":sys.version,"platform":platform.platform(),"run_mode":RUN_MODE,"upstream_commit":UPSTREAM_COMMIT}, indent=2))


In [ ]:
# Install the legacy Lightning generation required by the authors' two-optimizer training_step.
# Do not replace this with Lightning 2.x: it removed optimizer_idx and the old Trainer CLI APIs.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pytorch-lightning==1.9.5", "omegaconf==2.3.0",
                "torchmetrics==0.11.4", "einops", "easydict", "transformers==4.45.2"], check=True)
if not REPO.exists():
    subprocess.run(["git", "clone", UPSTREAM_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", UPSTREAM_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], check=True)
import numpy as np
if not hasattr(np, "Inf"):
    np.Inf = np.inf  # runtime compatibility for NumPy 2; does not modify upstream source
for p in (REPO / "adaptations", REPO / "adaptations" / "sqgan_div2k_channel"):
    p.mkdir(parents=True, exist_ok=True)
(REPO / "adaptations" / "__init__.py").write_text("", encoding="utf-8")
(REPO / "adaptations" / "sqgan_div2k_channel" / "__init__.py").write_text("", encoding="utf-8")


In [ ]:
# Keep the adaptation code versioned in this notebook; the cloned author checkout stays untouched.
CHANNEL_SOURCE = 'from __future__ import annotations\n\nimport torch\nfrom torch import nn\n\n\nclass ComplexLatentChannel(nn.Module):\n    """Per-example unit-power complex AWGN or flat Rayleigh channel.\n\n    Input follows the author VQ module layout [batch, real_feature_dim, tokens].\n    A 256-D real code vector is serialized as 128 complex channel uses.\n    This is a DIV2K/AWGN/Rayleigh adaptation, not the original SQ-GAN pipeline.\n    """\n    def __init__(self, channel: str, snr_db: float = 10.0, perfect_csi: bool = True):\n        super().__init__()\n        self.channel = channel.lower()\n        if self.channel not in {"awgn", "rayleigh"}:\n            raise ValueError(f"channel must be awgn or rayleigh, got {channel!r}")\n        self.snr_db = float(snr_db)\n        self.perfect_csi = bool(perfect_csi)\n\n    def forward(self, z: torch.Tensor) -> torch.Tensor:\n        if z.ndim != 3:\n            raise ValueError(f"Expected [B,C,N] latent tensor, got {tuple(z.shape)}")\n        b, c, n = z.shape\n        if c % 2:\n            raise ValueError(f"Real latent dimension must be even, got {c}")\n        pairs = z.transpose(1, 2).contiguous().reshape(b, n, c // 2, 2)\n        power = pairs.square().sum(dim=-1).mean(dim=(1, 2), keepdim=True).clamp_min(1e-12)\n        scaled = pairs / power.sqrt().unsqueeze(-1)\n        x = torch.complex(scaled[..., 0], scaled[..., 1])\n        noise_power = 10.0 ** (-self.snr_db / 10.0)\n        sigma = (noise_power / 2.0) ** 0.5\n        noise = torch.complex(torch.randn_like(x.real), torch.randn_like(x.real)) * sigma\n        if self.channel == "rayleigh":\n            h = torch.complex(torch.randn_like(x.real), torch.randn_like(x.real)) / (2.0 ** 0.5)\n            y = h * x + noise\n            if self.perfect_csi:\n                x_hat = y * h.conj() / h.abs().square().clamp_min(1e-8)\n            else:\n                x_hat = y\n        else:\n            x_hat = x + noise\n        received = torch.stack((x_hat.real, x_hat.imag), dim=-1) * power.sqrt().unsqueeze(-1)\n        return received.reshape(b, n, c).transpose(1, 2).contiguous()\n\n\nclass ChannelizedVectorQuantizer(nn.Module):\n    """Keep author VQ and insert a differentiable channel after its STE output."""\n    def __init__(self, quantizer_config: dict, channel: str, snr_db: float = 10.0,\n                 perfect_csi: bool = True):\n        super().__init__()\n        from utils.utils import instantiate_from_config\n        self.quantizer = instantiate_from_config(quantizer_config)\n        self.channel = ComplexLatentChannel(channel, snr_db, perfect_csi)\n\n    def forward(self, *args, **kwargs):\n        quantized, commitment_loss, info = self.quantizer(*args, **kwargs)\n        return self.channel(quantized), commitment_loss, info\n'
DATASET_SOURCE = 'from __future__ import annotations\n\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch.utils.data import Dataset\nfrom torchvision import transforms\n\n\nclass DIV2KWithPseudoLabels(Dataset):\n    """Paired RGB and 19-class Cityscapes-ID pseudo-labels at author\'s 2:1 geometry."""\n    def __init__(self, image_root: str, label_root: str, size=(256, 512), train: bool = True):\n        self.image_root = Path(image_root)\n        self.label_root = Path(label_root)\n        if isinstance(size, int):\n            self.height = self.width = int(size)\n        else:\n            self.height, self.width = map(int, size)\n        self.train = bool(train)\n        exts = {".png", ".jpg", ".jpeg"}\n        images = sorted(p for p in self.image_root.rglob("*") if p.suffix.lower() in exts)\n        if not images:\n            raise FileNotFoundError(f"No images under {self.image_root}")\n        self.pairs = []\n        missing = []\n        for image in images:\n            mask = self.label_root / f"{image.stem}.png"\n            if mask.is_file():\n                self.pairs.append((image, mask))\n            else:\n                missing.append(str(mask))\n        if missing:\n            raise FileNotFoundError(f"Missing {len(missing)} pseudo masks; first: {missing[0]}")\n        self.image_norm = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))\n\n    def __len__(self):\n        return len(self.pairs)\n\n    def _resize_cover(self, image, mask):\n        scale = max(self.width / image.width, self.height / image.height)\n        new_size = (max(self.width, round(image.width * scale)), max(self.height, round(image.height * scale)))\n        image = image.resize(new_size, Image.Resampling.BICUBIC)\n        mask = mask.resize(new_size, Image.Resampling.NEAREST)\n        return image, mask\n\n    def __getitem__(self, idx):\n        image_path, mask_path = self.pairs[idx]\n        image = Image.open(image_path).convert("RGB")\n        mask = Image.open(mask_path).convert("L")\n        image, mask = self._resize_cover(image, mask)\n        if self.train:\n            x = int(np.random.randint(0, image.width - self.width + 1))\n            y = int(np.random.randint(0, image.height - self.height + 1))\n        else:\n            x = (image.width - self.width) // 2\n            y = (image.height - self.height) // 2\n        box = (x, y, x + self.width, y + self.height)\n        image, mask = image.crop(box), mask.crop(box)\n        if self.train and np.random.rand() < 0.5:\n            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)\n            mask = mask.transpose(Image.Transpose.FLIP_LEFT_RIGHT)\n        image_tensor = self.image_norm(transforms.ToTensor()(image))\n        ids = torch.from_numpy(np.asarray(mask, dtype=np.uint8).copy()).long()\n        if torch.any((ids != 255) & (ids > 18)):\n            raise ValueError(f"Expected Cityscapes IDs [0,18] or 255: {mask_path}")\n        ids[ids == 255] = 19\n        semantic = torch.nn.functional.one_hot(ids, num_classes=20).permute(2, 0, 1).float()\n        label_ids = ids.clone()\n        label_ids[label_ids == 19] = 0\n        label = label_ids.unsqueeze(0).float()\n        return {"image": image_tensor, "label": label, "semantic": semantic, "filename": image_path.name}\n'
ADAPTER = REPO / "adaptations" / "sqgan_div2k_channel"
(ADAPTER / "channel.py").write_text(CHANNEL_SOURCE, encoding="utf-8")
(ADAPTER / "dataset.py").write_text(DATASET_SOURCE, encoding="utf-8")




# BEGIN SESSION ORCHESTRATION
COMPAT_DIR = Path("/tmp/sqgan_runtime")
COMPAT_DIR.mkdir(parents=True, exist_ok=True)
(COMPAT_DIR / 'session_control.py').write_text('"""Notebook orchestration for the pinned author Lightning 1.9 training script."""\nfrom __future__ import annotations\n\nimport json\nimport math\nimport os\nimport time\nfrom pathlib import Path\n\nimport pytorch_lightning as pl\nfrom pytorch_lightning.callbacks import Callback\n\n\nclass SessionControl(Callback):\n    def __init__(self):\n        super().__init__()\n        self.deadline = float(os.environ.get("SQGAN_SESSION_DEADLINE_UNIX", "inf"))\n        self.epoch_limit = int(os.environ.get("SQGAN_STOP_AFTER_STAGE_EPOCHS", "150"))\n        self.epoch_started = None\n\n    def on_train_start(self, trainer, pl_module):\n        expected = json.loads(os.environ.get("SQGAN_EXPECTED_RESUME", "null"))\n        rates = [[g["lr"] for g in opt.param_groups] for opt in trainer.optimizers]\n        actual = {"epoch_index": int(trainer.current_epoch), "global_step": int(trainer.global_step), "optimizer_learning_rates": rates}\n        if expected:\n            assert actual["global_step"] == expected["global_step"], (actual, expected)\n            assert len(trainer.optimizers) == expected["optimizer_states"], (actual, expected)\n            assert len(trainer.lr_scheduler_configs) == expected["lr_schedulers"], (actual, expected)\n            assert actual["epoch_index"] >= expected["saved_epoch_index"], (actual, expected)\n            for actual_groups, expected_groups in zip(rates, expected["optimizer_learning_rates"]):\n                assert len(actual_groups) == len(expected_groups)\n                assert all(math.isclose(a, b, rel_tol=1e-7, abs_tol=1e-12) for a, b in zip(actual_groups, expected_groups)), (actual, expected)\n            actual["resume_verified"] = True\n        else:\n            actual["resume_verified"] = False\n        if trainer.is_global_zero:\n            print("TRAINING STATE VERIFIED", json.dumps(actual), flush=True)\n            root = os.environ.get("SQGAN_STAGE_AUDIT_DIR")\n            if root:\n                destination = Path(root)\n                destination.mkdir(parents=True, exist_ok=True)\n                (destination / "resume_verified.json").write_text(json.dumps(actual, indent=2))\n\n    def on_train_epoch_start(self, trainer, pl_module):\n        self.epoch_started = time.time()\n\n    def on_train_epoch_end(self, trainer, pl_module, *args, **kwargs):\n        elapsed = time.time() - self.epoch_started if self.epoch_started else 0.0\n        completed = int(trainer.current_epoch) + 1\n        # Leave enough time for checkpoint serialization, mirroring and output upload.\n        local_stop = completed >= self.epoch_limit or time.time() + max(600, 1.5 * elapsed) >= self.deadline\n        stop = trainer.strategy.reduce_boolean_decision(local_stop, all=False)\n        trainer.should_stop = trainer.should_stop or stop\n        if trainer.is_global_zero:\n            record = {"completed_epochs": completed, "global_step": int(trainer.global_step), "epoch_seconds": elapsed, "session_seconds_remaining": max(0, self.deadline-time.time())}\n            for key, value in trainer.callback_metrics.items():\n                if getattr(value, "numel", lambda: 0)() == 1:\n                    record[key] = float(value.detach().cpu())\n            root = os.environ.get("SQGAN_STAGE_AUDIT_DIR")\n            if root:\n                with (Path(root) / "training_history.jsonl").open("a") as stream:\n                    stream.write(json.dumps(record) + "\\n")\n            print("EPOCH AUDIT", json.dumps(record), flush=True)\n            if stop:\n                print("NORMAL SESSION STOP REQUESTED at completed stage epoch", completed, flush=True)\n\n\ndef install():\n    if getattr(pl.Trainer, "_sqgan_session_control_installed", False):\n        return\n    original = pl.Trainer.fit\n\n    def fit(self, *args, **kwargs):\n        self.callbacks.append(SessionControl())\n        return original(self, *args, **kwargs)\n\n    pl.Trainer.fit = fit\n    pl.Trainer._sqgan_session_control_installed = True\n', encoding="utf-8")
(COMPAT_DIR / 'check_resume_contract.py').write_text('"""Verify Lightning checkpoint continuation on a tiny CPU model, not SQ-GAN."""\nimport json\nimport os\nfrom pathlib import Path\n\nimport numpy as np\nif not hasattr(np, "Inf"):\n    np.Inf = np.inf\nos.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"\n\nimport torch\nimport pytorch_lightning as pl\nfrom torch.utils.data import DataLoader, TensorDataset\nfrom session_control import install\n\n\nclass Probe(pl.LightningModule):\n    def __init__(self):\n        super().__init__()\n        self.generator = torch.nn.Linear(2, 1)\n        self.discriminator = torch.nn.Linear(2, 1)\n\n    def training_step(self, batch, batch_idx, optimizer_idx):\n        layer = self.generator if optimizer_idx == 0 else self.discriminator\n        return layer(batch[0]).square().mean()\n\n    def configure_optimizers(self):\n        opts = [torch.optim.Adam(layer.parameters(), lr=0.01) for layer in [self.generator, self.discriminator]]\n        return opts, [{"scheduler": torch.optim.lr_scheduler.StepLR(opt, step_size=2, gamma=0.5), "interval": "step"} for opt in opts]\n\n\ndef main(root):\n    root = Path(root)\n    root.mkdir(parents=True, exist_ok=True)\n    install()\n    os.environ["SQGAN_SESSION_DEADLINE_UNIX"] = "inf"\n    os.environ["SQGAN_STOP_AFTER_STAGE_EPOCHS"] = "2"\n    os.environ["SQGAN_EXPECTED_RESUME"] = "null"\n    os.environ["SQGAN_STAGE_AUDIT_DIR"] = str(root / "first")\n    loader = DataLoader(TensorDataset(torch.ones(8, 2)), batch_size=2)\n    def trainer():\n        return pl.Trainer(accelerator="cpu", devices=1, max_epochs=4, logger=False, enable_checkpointing=False, enable_progress_bar=False, enable_model_summary=False)\n    first = trainer()\n    first.fit(Probe(), loader)\n    checkpoint = root / "probe.ckpt"\n    first.save_checkpoint(checkpoint)\n    raw = torch.load(checkpoint, map_location="cpu", weights_only=False)\n    assert raw["epoch"] == 2 or raw["epoch"] == 1\n    expected = {"saved_epoch_index": raw["epoch"], "global_step": raw["global_step"], "optimizer_states":len(raw["optimizer_states"]),"lr_schedulers":len(raw["lr_schedulers"]),"optimizer_learning_rates":[[g["lr"] for g in opt["param_groups"]] for opt in raw["optimizer_states"]]}\n    os.environ["SQGAN_EXPECTED_RESUME"] = json.dumps(expected)\n    os.environ["SQGAN_STOP_AFTER_STAGE_EPOCHS"] = "4"\n    os.environ["SQGAN_STAGE_AUDIT_DIR"] = str(root / "second")\n    second = trainer()\n    second.fit(Probe(), loader, ckpt_path=str(checkpoint))\n    assert second.current_epoch == 4, second.current_epoch\n    assert second.global_step > raw["global_step"]\n    os.environ["SQGAN_EXPECTED_RESUME"] = "null"\n    os.environ["SQGAN_SESSION_DEADLINE_UNIX"] = "0"\n    os.environ["SQGAN_STAGE_AUDIT_DIR"] = str(root / "deadline")\n    timed = trainer()\n    timed.fit(Probe(), loader)\n    assert timed.current_epoch == 1, timed.current_epoch\n    timed.save_checkpoint(root / "deadline_stop.ckpt")\n    timed_state = torch.load(root / "deadline_stop.ckpt", map_location="cpu", weights_only=False)\n    assert len(timed_state["optimizer_states"]) == len(timed_state["lr_schedulers"]) == 2\n    report = {"result":"PASS", "initial_global_step":raw["global_step"],"resumed_final_global_step":second.global_step,"resumed_final_epoch":second.current_epoch,"optimizer_and_scheduler_states_restored":True,"deadline_stop_and_checkpoint_verified":True}\n    (root / "resume_contract_pass.json").write_text(json.dumps(report, indent=2))\n    print("RESUME CONTRACT PASS", json.dumps(report), flush=True)\n\n\nif __name__ == "__main__":\n    import sys\n    main(sys.argv[1])\n', encoding="utf-8")

# Restore only the exported training state; the official source was freshly cloned above.
for input_root in Path("/kaggle/input").iterdir():
    for prior_out in input_root.rglob("sqgan_div2k_channel_results"):
        if prior_out.is_dir() and (prior_out / "stage_state").is_dir():
            shutil.copytree(prior_out, OUTPUT, dirs_exist_ok=True)
            print("RESTORED SAVED OUTPUT", prior_out, flush=True)
    for prior_logs in input_root.rglob("upstream_logs"):
        if prior_logs.is_dir():
            shutil.copytree(prior_logs, REPO / "logs", dirs_exist_ok=True)
            print("RESTORED UPSTREAM LOGS", prior_logs, flush=True)


## Compatibility gate

This stage must succeed before the long training loop is enabled. It checks the exact legacy trainer APIs, mounts the expected DIV2K/Kodak inputs, builds the `all` model with channelized image and semantic quantizers, then verifies shape and gradient flow through one 256×512 forward/backward pass. The author config’s `ratio_dim=2`, latent grid 16×32, model layers, and training implementation are retained.


In [ ]:
sys.path.insert(0, str(REPO))
os.chdir(REPO)
import torch, pytorch_lightning as pl
import pytorch_lightning.plugins as pl_plugins
from pytorch_lightning.strategies import DDPStrategy
# Runtime compatibility shim: preserve the author import/API call without editing train.py.
pl_plugins.DDPPlugin = DDPStrategy
from pytorch_lightning.plugins import DDPPlugin
from pytorch_lightning import Trainer
from omegaconf import OmegaConf
from utils.utils import instantiate_from_config
import train as author_train

assert pl.__version__ == "1.9.5", f"Expected the Kaggle-validated legacy Lightning 1.9.5 API, got {pl.__version__}"
assert hasattr(Trainer, "add_argparse_args") and hasattr(Trainer, "from_argparse_args"), "Author train.py CLI API unavailable"
assert hasattr(author_train, "get_parser") and DDPPlugin is not None
# Verify the upstream train.py constructor now resolves to the supported strategy class.
assert isinstance(DDPPlugin(find_unused_parameters=True), DDPStrategy)
assert torch.cuda.is_available(), "Kaggle GPU is required for this model preflight"
print({"python":sys.version,"torch":torch.__version__,"lightning":pl.__version__,"cuda":torch.version.cuda,"gpus":[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]})
for p in (DIV2K_TRAIN, DIV2K_VAL, KODAK_ROOT):
    print("input", p, "exists=",p.exists(),"sample_files=",sum(1 for q in p.rglob("*") if q.suffix.lower() in {".png",".jpg",".jpeg"}) if p.exists() else 0)
    assert p.exists(), f"Missing mounted dataset: {p}"

cfg = OmegaConf.load(BASE_CONFIG)
assert IMAGE_HEIGHT == 256 and IMAGE_WIDTH == 512, "DIV2K crop must preserve author 2:1 geometry"
for key in ("masker_config", "masker_semantic_config", "demasker_config", "demasker_semantic_config"):
    assert int(cfg.model.params[key].params.ratio_dim) == 2, f"Unexpected upstream geometry in {key}"
for key in ("vqconfig", "vqconfig_semantic"):
    original = OmegaConf.to_container(cfg.model.params[key], resolve=True)
    cfg.model.params[key] = OmegaConf.create({
        "target":"adaptations.sqgan_div2k_channel.channel.ChannelizedVectorQuantizer",
        "params":{"quantizer_config":original,"channel":"awgn","snr_db":SNR_DB,"perfect_csi":PERFECT_CSI}})
cfg.model.params.mode = "all"
# Released image-loss YAML contains 20 weights, but loss masks semantic[:, :-1] (19 classes).
cfg.model.params.lossconfig.params.label_weights = cfg.model.params.lossconfig.params.label_weights[:19]
model = instantiate_from_config(cfg.model).cuda().eval()
image = torch.randn(1,3,IMAGE_HEIGHT,IMAGE_WIDTH,device="cuda")
ids = torch.randint(0,19,(1,IMAGE_HEIGHT,IMAGE_WIDTH),device="cuda")
semantic = torch.nn.functional.one_hot(ids,num_classes=20).permute(0,3,1,2).float()
label = ids.unsqueeze(1).float()
start = time.time()
model.train()
smoke_batch={"image":image,"label":label,"semantic":semantic}
loss_values=[]
with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU,torch.profiler.ProfilerActivity.CUDA], with_flops=True) as prof:
    for optimizer_idx in (0,1):
        model.zero_grad(set_to_none=True)
        loss=model.training_step(smoke_batch,0,optimizer_idx)
        assert torch.isfinite(loss).all(), f"Nonfinite training loss for optimizer {optimizer_idx}: {loss}"
        loss_values.append(float(loss.detach()))
        loss.backward()
torch.cuda.synchronize()
flops = sum(int(getattr(evt,"flops",0) or 0) for evt in prof.key_averages())
assert len(loss_values)==2 and all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None), "No finite gradients from both training optimizers"
model.eval()
with torch.inference_mode():
    out=model.forward_all(image,semantic)
assert out[0].shape == image.shape == (1,3,256,512), f"Unexpected image reconstruction shape: {out[0].shape}"
assert out[3].shape == semantic.shape, f"Unexpected semantic reconstruction shape: {out[3].shape}"
report={"preflight":"PASS","upstream_commit":UPSTREAM_COMMIT,"lightning":pl.__version__,"torch":torch.__version__,"gpus":[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],"image_reconstruction_shape":list(out[0].shape),"semantic_reconstruction_shape":list(out[3].shape),"smoke_train_losses_optimizer_0_1":loss_values,"profiler_counted_flops_two_optimizer_forward_backward":flops,"elapsed_seconds":round(time.time()-start,2),"peak_cuda_memory_bytes":torch.cuda.max_memory_allocated(),"synthetic_smoke_batch_only":True}
(OUTPUT/"preflight_report.json").write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))
del model, out, image, semantic, label, ids, prof, smoke_batch
torch.cuda.empty_cache()




# BEGIN RESUME CONTRACT CHECK
# Exercise a real two-optimizer checkpoint/restore on a tiny CPU-only test model.
probe_env = os.environ.copy()
probe_env["PYTHONPATH"] = str(COMPAT_DIR) + os.pathsep + probe_env.get("PYTHONPATH", "")
subprocess.run([sys.executable, str(COMPAT_DIR/"check_resume_contract.py"), str(OUTPUT/"preflight_resume_contract")], env=probe_env, check=True)


## Training with verified continuation and a session deadline

The official source remains pinned and unmodified. DIV2K pseudo-segmentation and the noisy channel are explicit adaptations. Each author stage retains its 150-epoch target and original optimizer/scheduler setup. The three stages run independently for AWGN and Rayleigh.

A runtime callback verifies restored optimizer learning rates and global step, writes epoch audit records, and requests a normal stop before the 9.5-hour notebook budget ends. It does not alter the model forward path, gradients, losses, or optimizer updates. A CPU-only two-optimizer test must first demonstrate a successful stop/checkpoint/restore.

The latest complete Lightning checkpoint and configs from each stage are mirrored atomically into the output folder. Repository files, top-k duplicates and diagnostic images stay in `/tmp`. Successful output from the preceding chunk must be attached as a notebook input. Completed stages are skipped; partial stages resume through the author's `--resume` path. Evaluation runs only after all six stages are complete and enough session time remains.


In [ ]:
if RUN_MODE != "full":
    print("Preflight complete; full training is disabled. Set RUN_MODE='full' after reviewing the compatibility output.")
else:
    from transformers import AutoImageProcessor, SegformerForSemanticSegmentation
    import torch.nn.functional as F
    from PIL import Image
    import numpy as np

    LABEL_MODEL = "nvidia/segformer-b0-finetuned-cityscapes-1024-1024"
    processor = AutoImageProcessor.from_pretrained(LABEL_MODEL)
    segmenter = SegformerForSemanticSegmentation.from_pretrained(LABEL_MODEL).cuda().eval()
    LABEL_REVISION = getattr(segmenter.config, "_commit_hash", None)
    assert LABEL_REVISION, "Could not resolve immutable SegFormer revision"
    def make_pseudo_labels(image_root, label_root):
        image_root, label_root = Path(image_root), Path(label_root)
        label_root.mkdir(parents=True, exist_ok=True)
        files = sorted(p for p in image_root.rglob("*") if p.suffix.lower() in {".png",".jpg",".jpeg"})
        assert files, f"No images under {image_root}"
        with torch.inference_mode():
            for i,p in enumerate(files,1):
                out = label_root / (p.stem+".png")
                if out.exists(): continue
                im = Image.open(p).convert("RGB"); w,h=im.size
                x = processor(images=im,return_tensors="pt").to("cuda")
                logits=segmenter(**x).logits
                ids=F.interpolate(logits,size=(h,w),mode="bilinear",align_corners=False).argmax(1)[0].byte().cpu().numpy()
                Image.fromarray(ids,mode="L").save(out)
                if i%50==0 or i==len(files): print(f"pseudo labels {i}/{len(files)}")
        return len(files)
    train_labels=OUTPUT/"pseudo_labels"/"div2k_train"
    val_labels=OUTPUT/"pseudo_labels"/"div2k_val"
    kodak_labels=OUTPUT/"pseudo_labels"/"kodak"
    n_train=make_pseudo_labels(DIV2K_TRAIN,train_labels)
    n_val=make_pseudo_labels(DIV2K_VAL,val_labels)
    n_kodak=make_pseudo_labels(KODAK_ROOT,kodak_labels)
    (OUTPUT/"pseudo_label_metadata.json").write_text(json.dumps({"model_id":LABEL_MODEL,"revision":LABEL_REVISION,"train":n_train,"validation":n_val,"kodak":n_kodak},indent=2))
    print("Pseudo labels ready", n_train,n_val,n_kodak,"revision",LABEL_REVISION)


In [ ]:
if RUN_MODE == "full":
    import glob
    from omegaconf import OmegaConf
    from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure, MultiScaleStructuralSimilarityIndexMeasure
    from torchvision.transforms.functional import to_tensor
    import csv
    from adaptations.sqgan_div2k_channel.dataset import DIV2KWithPseudoLabels

    def channel_config(channel, mode, parts=None):
        c=OmegaConf.load(BASE_CONFIG)
        for key in ("vqconfig","vqconfig_semantic"):
            base=OmegaConf.to_container(c.model.params[key],resolve=True)
            c.model.params[key]=OmegaConf.create({"target":"adaptations.sqgan_div2k_channel.channel.ChannelizedVectorQuantizer","params":{"quantizer_config":base,"channel":channel,"snr_db":SNR_DB,"perfect_csi":PERFECT_CSI}})
        c.model.params.mode=mode
        c.model.params.lossconfig.params.label_weights=c.model.params.lossconfig.params.label_weights[:19]
        c.model.params.ckpt_path_parts=str(parts) if parts else None
        c.data.params.batch_size=BATCH_SIZE
        c.data.params.num_workers=0  # avoid DDP child-loader instability on Kaggle Python 3.13
        c.data.params.train={"target":"adaptations.sqgan_div2k_channel.dataset.DIV2KWithPseudoLabels","params":{"image_root":str(DIV2K_TRAIN),"label_root":str(train_labels),"size":(IMAGE_HEIGHT,IMAGE_WIDTH),"train":True}}
        c.data.params.validation={"target":"adaptations.sqgan_div2k_channel.dataset.DIV2KWithPseudoLabels","params":{"image_root":str(DIV2K_VAL),"label_root":str(val_labels),"size":(IMAGE_HEIGHT,IMAGE_WIDTH),"train":False}}
        return c

    def split_trained_components(ckpt_path, outdir, names):
        raw=torch.load(ckpt_path,map_location="cpu",weights_only=False)
        state=raw["state_dict"]
        outdir.mkdir(parents=True,exist_ok=True)
        for name in names:
            prefix=name+"."
            part={k[len(prefix):]:v for k,v in state.items() if k.startswith(prefix)}
            assert part, f"No trained {name} parameters in {ckpt_path}"
            torch.save(part,outdir/(name+".ckpt"))
            print("extracted",name,len(part),"tensors")

    def latest_last_ckpt(channel,mode):
        candidates=list((REPO/"logs").glob(f"*sqgan_{channel}_{mode}*/checkpoints/last*.ckpt"))
        assert candidates, f"No last.ckpt found for {channel}/{mode}"
        return max(candidates,key=lambda p:(p.stat().st_mtime_ns,p.name=="last.ckpt"))

    import gc
    import threading

    def checkpoint_info(path):
        raw = torch.load(path, map_location="cpu", weights_only=False)
        progress = raw.get("loops", {}).get("fit_loop", {}).get("epoch_progress", {})
        result = {
            "saved_epoch_index": int(raw["epoch"]),
            "completed_epochs": int(progress.get("total", {}).get("completed", raw["epoch"])),
            "global_step": int(raw["global_step"]),
            "optimizer_states": len(raw.get("optimizer_states", [])),
            "lr_schedulers": len(raw.get("lr_schedulers", [])),
            "optimizer_learning_rates": [[g["lr"] for g in opt["param_groups"]] for opt in raw.get("optimizer_states", [])],
            "state_dict_tensors": len(raw["state_dict"]),
        }
        assert result["optimizer_states"] == result["lr_schedulers"] == 2, result
        del raw
        gc.collect()
        return result

    for split_name, image_root, label_root in (("train",DIV2K_TRAIN,train_labels),("validation",DIV2K_VAL,val_labels),("kodak",KODAK_ROOT,kodak_labels)):
        images = [p for p in Path(image_root).rglob("*") if p.suffix.lower() in {".png",".jpg",".jpeg"}]
        assert images and all((Path(label_root)/(p.stem+".png")).is_file() for p in images), split_name
        print("DATASET CHECK PASS", split_name, len(images), flush=True)

    compat_dir = COMPAT_DIR
    (compat_dir / "sitecustomize.py").write_text('import os, sys\nimport numpy as np\nif not hasattr(np, "Inf"): np.Inf=np.inf\nsys.path.insert(0,os.getcwd())\nimport pytorch_lightning.plugins as _plugins\nfrom pytorch_lightning.strategies import DDPStrategy\n_plugins.DDPPlugin=DDPStrategy\nfrom utils.logger import SetupCallback, CaptionImageLogger\nSetupCallback.on_fit_start=SetupCallback.on_pretrain_routine_start\ndelattr(SetupCallback,"on_pretrain_routine_start")\n_upstream_tensorboard=CaptionImageLogger._tensorboard\ndef _tensorboard(self,pl_module,images,batch_idx,split):\n    safe={k:v for k,v in images.items() if k not in ("ordered_vqidk","semantic_ordered_vqidk")}\n    return _upstream_tensorboard(self,pl_module,safe,batch_idx,split)\nCaptionImageLogger._tensorboard=_tensorboard\nfrom session_control import install\ninstall()\n', encoding="utf-8")
    child_env = os.environ.copy()
    child_env["PYTHONPATH"] = str(compat_dir) + os.pathsep + child_env.get("PYTHONPATH", "")
    child_env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
    child_env["SQGAN_STOP_AFTER_STAGE_EPOCHS"] = str(STOP_AFTER_STAGE_EPOCHS)
    child_env["SQGAN_SESSION_DEADLINE_UNIX"] = str(SESSION_DEADLINE_UNIX)
    child_env["SQGAN_OUTPUT_DIR"] = str(OUTPUT)

    source_diff = subprocess.run(["git","diff","--exit-code",UPSTREAM_COMMIT,"--"],cwd=REPO,capture_output=True,text=True)
    assert source_diff.returncode == 0, source_diff.stdout + source_diff.stderr
    assert subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip() == UPSTREAM_COMMIT
    print("UPSTREAM SOURCE INTEGRITY PASS", UPSTREAM_COMMIT, flush=True)
    # Release all GPU tensors from pseudo-labelling before the distributed training subprocess.
    del segmenter, processor
    gc.collect()
    torch.cuda.empty_cache()

    mirror_stop = threading.Event()
    mirror_lock = threading.Lock()
    mirrored_signatures = {}
    def mirror_checkpoints(force=False):
        logs_root = REPO / "logs"
        if not logs_root.exists(): return
        # Lightning may use last-v1.ckpt after restoring into a different log directory.
        # Export only the newest variant per stage under the canonical last.ckpt name.
        newest = {}
        for candidate in logs_root.glob("*sqgan_*_150ep/checkpoints/last*.ckpt"):
            key = candidate.parent
            if key not in newest or (candidate.stat().st_mtime_ns, candidate.name == "last.ckpt") > (newest[key].stat().st_mtime_ns, newest[key].name == "last.ckpt"):
                newest[key] = candidate
        candidates = list(newest.values()) + list(logs_root.glob("*sqgan_*_150ep/configs/*.yaml"))
        for src in candidates:
            try:
                stat = src.stat()
                signature = (stat.st_size, stat.st_mtime_ns)
                previous = mirrored_signatures.get(str(src))
                mirrored_signatures[str(src)] = signature
                if not force and previous != signature: continue
                relative = src.relative_to(logs_root)
                if src.name.startswith("last") and src.suffix == ".ckpt": relative = relative.with_name("last.ckpt")
                dst = OUTPUT / "upstream_logs" / relative
                if dst.exists() and dst.stat().st_size == stat.st_size and dst.stat().st_mtime_ns >= stat.st_mtime_ns: continue
                dst.parent.mkdir(parents=True,exist_ok=True)
                temporary = dst.with_name(dst.name+".partial")
                shutil.copy2(src,temporary)
                after = src.stat()
                if (after.st_size,after.st_mtime_ns) == signature and temporary.stat().st_size == stat.st_size:
                    os.replace(temporary,dst)
                    print("CHECKPOINT EXPORTED", str(dst.relative_to(OUTPUT)), stat.st_size, flush=True)
            except (OSError,FileNotFoundError) as exc:
                print("CHECKPOINT EXPORT RETRY", src.name, repr(exc), flush=True)

    _mirror_without_lock = mirror_checkpoints
    def mirror_checkpoints(force=False):
        with mirror_lock:
            _mirror_without_lock(force)

    def mirror_loop():
        while not mirror_stop.is_set():
            mirror_checkpoints()
            mirror_stop.wait(20)

    mirror_thread = threading.Thread(target=mirror_loop,daemon=True)
    mirror_thread.start()
    stage_state_dir = OUTPUT / "stage_state"
    stage_state_dir.mkdir(exist_ok=True)
    chunk_paused = False
    try:
        for channel in CHANNELS:
            channel_out = OUTPUT / channel
            channel_out.mkdir(exist_ok=True)
            parts = channel_out / "components"
            for mode in ("ssm","img","all"):
                marker = stage_state_dir / f"{channel}_{mode}.json"
                if marker.is_file() and json.loads(marker.read_text()).get("status") == "complete":
                    print("STAGE ALREADY COMPLETE",channel,mode,flush=True)
                    continue
                if time.time()+1200 >= SESSION_DEADLINE_UNIX:
                    chunk_paused=True
                    break
                cfg=channel_config(channel,mode,parts if mode=="all" else None)
                cfg_path=OUTPUT/f"config_{channel}_{mode}.yaml"
                OmegaConf.save(cfg,cfg_path)
                args=[sys.executable,"train.py","--base",str(cfg_path),"--mode",mode,"--max_epochs",str(EPOCHS_PER_STAGE),"--gpus","0,1","--precision","32","--logtype","tensorboard","--check_val_every_n_epoch","1","--num_sanity_val_steps","0"]
                candidates=list((REPO/"logs").glob(f"*sqgan_{channel}_{mode}_150ep/checkpoints/last.ckpt"))
                stage_env=child_env.copy()
                stage_env["SQGAN_STAGE_AUDIT_DIR"]=str(OUTPUT/"audit"/f"{channel}_{mode}")
                if candidates:
                    resume=max(candidates,key=lambda p:p.stat().st_mtime)
                    expected=checkpoint_info(resume)
                    stage_env["SQGAN_EXPECTED_RESUME"]=json.dumps(expected)
                    args += ["--resume",str(resume.parents[1])]
                    print("RESUMING AUTHOR CHECKPOINT",channel,mode,json.dumps(expected),flush=True)
                else:
                    stage_env["SQGAN_EXPECTED_RESUME"]="null"
                    args += ["--name",f"sqgan_{channel}_{mode}_150ep"]
                if mode=="all": args += ["--ckpt_path_parts",str(parts)]
                print("TRAIN",channel,mode," ".join(args),flush=True)
                subprocess.run(args,cwd=REPO,env=stage_env,check=True)
                ckpt=latest_last_ckpt(channel,mode)
                info=checkpoint_info(ckpt)
                mirror_checkpoints(force=True)
                exported=OUTPUT/"upstream_logs"/ckpt.relative_to(REPO/"logs").with_name("last.ckpt")
                assert exported.is_file() and exported.stat().st_size==ckpt.stat().st_size
                info.update({"channel":channel,"mode":mode,"target_epochs":EPOCHS_PER_STAGE,"checkpoint_relative":str(exported.relative_to(OUTPUT)),"bytes":exported.stat().st_size,"status":"complete" if info["completed_epochs"]>=EPOCHS_PER_STAGE else "partial","updated_unix":time.time()})
                marker.write_text(json.dumps(info,indent=2))
                print("SAVED STAGE STATE",json.dumps(info),flush=True)
                if info["status"] != "complete":
                    chunk_paused=True
                    break
                if mode=="ssm": split_trained_components(ckpt,parts,["encoder_semantic","decoder_semantic","masker_semantic","demasker_semantic","quantize_semantic"])
                elif mode=="img": split_trained_components(ckpt,parts,["encoder","decoder","masker","demasker","quantize"])
                else: (channel_out/"final_checkpoint.json").write_text(json.dumps(info,indent=2))
            if chunk_paused: break
    finally:
        mirror_stop.set()
        mirror_thread.join(timeout=120)
        mirror_checkpoints(force=True)
    states=[json.loads(p.read_text()) for p in stage_state_dir.glob("*.json")]
    training_complete=len(states)==6 and all(p.get("status")=="complete" and p.get("completed_epochs",0)>=EPOCHS_PER_STAGE for p in states)
    manifest={"upstream_commit":UPSTREAM_COMMIT,"model":"SQ-GAN-DIV2K-channel-adaptation","session_budget_hours":SESSION_HOURS,"channels":CHANNELS,"snr_db":SNR_DB,"epochs_per_stage":EPOCHS_PER_STAGE,"stages":states,"training_complete":training_complete,"comparison_ready":False,"exported_files":[{"path":str(p.relative_to(OUTPUT)),"bytes":p.stat().st_size} for p in OUTPUT.rglob("*") if p.is_file()]}
    (OUTPUT/"run_manifest.json").write_text(json.dumps(manifest,indent=2))
    print("ALL TRAINING COMPLETE" if training_complete else "CHUNK SAVED NORMALLY; CONTINUATION REQUIRED",json.dumps(states),flush=True)


## Results and evaluation

After both channel loops finish, the checkpoint files and stage logs are in the Kaggle output directory `sqgan_div2k_channel_results/`. The model has two communicated streams: selected semantic VQ vectors and selected image VQ vectors. For each 256-D vector, the adapter uses 128 complex channel uses. Report actual selected token counts for each stream; the serialized 10-bit VQ indices, mask positions, and other control bits are separate side information and are not added to analog channel-use CBR. Do not compare that channel-use CBR as if it were the original SQ-GAN’s digital bit rate.

FLOP counts from `torch.profiler` cover operations for which the installed PyTorch profiler reports FLOPs; unsupported fused/custom operations may be omitted. Training FLOPs should be measured from a representative forward+backward pass, inference FLOPs from `forward_all` under no-grad.


In [ ]:
if RUN_MODE == "full" and training_complete and time.time()+1800 < SESSION_DEADLINE_UNIX:
    import math
    from torch.utils.data import DataLoader
    from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure, MultiScaleStructuralSimilarityIndexMeasure

    mean=torch.tensor([0.485,0.456,0.406],device="cuda").view(1,3,1,1)
    std=torch.tensor([0.229,0.224,0.225],device="cuda").view(1,3,1,1)
    def to_rgb01(x): return ((x*std+mean).clamp(0,1))
    results=[]; flops_rows=[]
    for channel in CHANNELS:
        channel_out=OUTPUT/channel
        final_ckpt=OUTPUT/json.loads((channel_out/"final_checkpoint.json").read_text())["checkpoint_relative"]
        assert final_ckpt.is_file(),f"Missing final checkpoint {final_ckpt}"
        cfg=channel_config(channel,"all",channel_out/"components")
        cfg.model.params.ckpt_path_parts=str(channel_out/"components")
        eval_model=instantiate_from_config(cfg.model).cuda().eval()
        eval_model.load_state_dict(torch.load(final_ckpt,map_location="cpu",weights_only=False)["state_dict"],strict=False)
        testset=DIV2KWithPseudoLabels(str(KODAK_ROOT),str(kodak_labels),size=(IMAGE_HEIGHT,IMAGE_WIDTH),train=False)
        assert len(testset)==24,f"Expected Kodak-24, found {len(testset)} images"
        loader=DataLoader(testset,batch_size=1,shuffle=False,num_workers=2)
        psnr=PeakSignalNoiseRatio(data_range=1.0).cuda()
        ssim=StructuralSimilarityIndexMeasure(data_range=1.0).cuda()
        msssim=MultiScaleStructuralSimilarityIndexMeasure(data_range=1.0).cuda()
        channel_rows=[]
        for idx,batch in enumerate(loader):
            torch.manual_seed(4100+idx); torch.cuda.manual_seed_all(4100+idx)
            x=batch["image"].cuda(); sem=batch["semantic"].cuda()
            with torch.inference_mode():
                out=eval_model.forward_all(x,sem)
                recon=to_rgb01(out[0]); ref=to_rgb01(x)
            a=psnr(ref,recon); b=ssim(ref,recon); c=msssim(ref,recon)
            n_img=int(out[6]["sampled_length"]); n_sem=int(out[7]["sampled_length"])
            uses_img=n_img*(256//2); uses_sem=n_sem*(256//2); uses=uses_img+uses_sem
            h,w=ref.shape[-2:]
            row={"model":"SQ-GAN-adaptation","channel":channel,"snr_db":SNR_DB,"filename":batch["filename"][0],"psnr_db":float(a),"ssim":float(b),"ms_ssim":float(c),"selected_image_tokens":n_img,"selected_semantic_tokens":n_sem,"image_complex_uses":uses_img,"semantic_complex_uses":uses_sem,"total_complex_uses":uses,"latent_cbr_complex_uses_per_rgb_value":uses/(3*h*w),"coded_vq_index_bits_lower_bound":10*(n_img+n_sem),"mask_position_bits_counted":False,"image_size":f"{h}x{w}"}
            results.append(row); channel_rows.append(row)
            if (idx+1)%6==0: print(channel, f"Kodak {idx+1}/24")
        summary={"model":"SQ-GAN-adaptation","channel":channel,"snr_db":SNR_DB,"images":len(channel_rows),"psnr_db":sum(r["psnr_db"] for r in channel_rows)/len(channel_rows),"ssim":sum(r["ssim"] for r in channel_rows)/len(channel_rows),"ms_ssim":sum(r["ms_ssim"] for r in channel_rows)/len(channel_rows),"complex_uses_per_image":sum(r["total_complex_uses"] for r in channel_rows)/len(channel_rows),"latent_cbr_complex_uses_per_rgb_value":sum(r["latent_cbr_complex_uses_per_rgb_value"] for r in channel_rows)/len(channel_rows)}
        import pandas as pd
        pd.DataFrame(channel_rows).to_csv(channel_out/"kodak_per_image_metrics.csv",index=False)
        (channel_out/"kodak_summary.json").write_text(json.dumps(summary,indent=2))
        print(json.dumps(summary,indent=2))

        # Profile one Kodak-shaped sample. Counts are profiler-reported FLOPs, not wall-clock throughput.
        example=next(iter(loader)); x=example["image"].cuda(); sem=example["semantic"].cuda()
        torch.cuda.reset_peak_memory_stats()
        with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU,torch.profiler.ProfilerActivity.CUDA],with_flops=True) as prof:
            with torch.inference_mode(): _=eval_model.forward_all(x,sem)
        torch.cuda.synchronize(); inf_flops=sum(int(getattr(e,"flops",0) or 0) for e in prof.key_averages())
        del prof
        eval_model.train(); eval_model.zero_grad(set_to_none=True); torch.cuda.reset_peak_memory_stats()
        eval_model.train()
        # Match the official final-stage freeze when profiling training forward/backward.
        for name in ("encoder_semantic","decoder_semantic","masker_semantic","demasker_semantic","quantize_semantic"):
            for parameter in getattr(eval_model,name).parameters(): parameter.requires_grad=False
        train_batch={"image":x,"label":example["label"].cuda(),"semantic":sem}
        with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU,torch.profiler.ProfilerActivity.CUDA],with_flops=True) as prof:
            for optimizer_idx in (0,1):
                eval_model.zero_grad(set_to_none=True)
                train_loss=eval_model.training_step(train_batch,0,optimizer_idx)
                train_loss.backward()
        torch.cuda.synchronize(); train_flops=sum(int(getattr(e,"flops",0) or 0) for e in prof.key_averages())
        del prof,train_batch,train_loss; torch.cuda.empty_cache(); eval_model.eval()
        flops_rows.extend([{"channel":channel,"profile":"inference_forward_all","profiler_counted_flops_per_256x512_image":inf_flops,"includes_backward":False,"includes_adversarial_loss":False},{"channel":channel,"profile":"training_author_two_optimizer_steps_forward_backward","profiler_counted_flops_per_256x512_image":train_flops,"includes_backward":True,"includes_adversarial_loss":True}])
        print("FLOPs profile",channel,"inference",inf_flops,"train forward/backward",train_flops)
        torch.save(eval_model.state_dict(),OUTPUT/f"SQGAN_{channel}_model.pth")
        del eval_model
        torch.cuda.empty_cache()
    import pandas as pd
    pd.DataFrame(results).to_csv(OUTPUT/"sqgan_kodak_per_image_metrics.csv",index=False)
    pd.DataFrame(flops_rows).to_csv(OUTPUT/"sqgan_flops_profile.csv",index=False)
    print("Wrote",OUTPUT/"sqgan_kodak_per_image_metrics.csv",OUTPUT/"sqgan_flops_profile.csv")

    manifest["comparison_ready"]=True
    manifest["evaluation_complete"]=True
    (OUTPUT/"run_manifest.json").write_text(json.dumps(manifest,indent=2))
